In [64]:
import os
import sys
import logging
import hashlib
import importlib
from pathlib import Path
import numpy as np
from datetime import datetime
from typing import Optional, Callable, Dict, List, Any
from astropy.io import fits

In [65]:
logger = logging.getLogger(__name__)

def _import_discovered_module(search_dirs: List[Path], file_pattern: str, required_text: Optional[str] = None):
    last_error = None

    for search_dir in search_dirs:
        if not search_dir.exists():
            continue

        search_dir_str = str(search_dir)
        if search_dir_str not in sys.path:
            sys.path.insert(0, search_dir_str)

        for module_path in search_dir.rglob(file_pattern):
            if required_text is not None:
                try:
                    module_text = module_path.read_text(encoding="utf-8", errors="ignore")
                except OSError:
                    continue

                if required_text not in module_text:
                    continue

            if module_path.name == "__init__.py":
                module_parts = module_path.parent.relative_to(search_dir).parts
            else:
                module_parts = module_path.relative_to(search_dir).with_suffix("").parts

            if not module_parts:
                continue

            module_name = ".".join(module_parts)

            try:
                return importlib.import_module(module_name)
            except ModuleNotFoundError as exc:
                last_error = exc

    raise ModuleNotFoundError(f"Could not resolve local module matching '{file_pattern}'") from last_error


In [66]:
_current_dir = Path.cwd().resolve()
_project_root = next(
    (candidate for candidate in (_current_dir, *_current_dir.parents) if (candidate / "notebooks").exists()),
    _current_dir,
)
_search_dirs = [_project_root, _project_root / "src"]

models_module = _import_discovered_module(_search_dirs, "models/__init__.py")
masters_module = importlib.import_module(f"{models_module.__name__}.masters")
utils_module = _import_discovered_module(_search_dirs, "utils.py", required_text="def fits_image_data")

FitsFileModel = getattr(models_module, "fitsFile")
FitsSessionModel = getattr(models_module, "fitsSession")
Masters = getattr(masters_module, "Masters")
normalize_file_path = getattr(utils_module, "normalize_file_path")
_fits_image_data = getattr(utils_module, "fits_image_data")

In [67]:
def _update_calibrated_frame_header(header, calibration_steps, bias_master, dark_master, 
                                   flat_master, calibrated_data, light_path):
    """
    Update FITS header of calibrated light frame with comprehensive metadata.
    
    Adds astronomy-standard headers for calibrated light frames including:
    - Calibration identification and status
    - Master frame references with full paths and checksums
    - Processing history and timestamps
    - Quality metrics and statistics
    - Software version and processing parameters
    
    Args:
        header: FITS header object to update
        calibration_steps: List of calibration steps applied
        bias_master: Path to bias master (or None)
        dark_master: Path to dark master (or None)
        flat_master: Path to flat master (or None)
        calibrated_data: Calibrated image data array
        light_path: Original light frame path
    """
    import numpy as np
    import hashlib
    import os
    from datetime import datetime
    
    # =================================================================
    # PRIMARY CALIBRATION IDENTIFICATION
    # =================================================================
    header['CALIBRAT'] = (True, 'Image has been calibrated')
    header['IMAGETYP'] = ('LIGHT_CAL', 'Calibrated light frame')
    header['CALDATE'] = (datetime.now().isoformat(), 'Calibration processing timestamp')
    header['CALSOFT'] = ('AstroFiler v1.2.0', 'Calibration software and version')
    
    # Original file reference
    header['ORIGFILE'] = (os.path.basename(light_path), 'Original uncalibrated filename')
    header['ORIGPATH'] = (light_path, 'Full path to original file')
    
    # =================================================================
    # CALIBRATION PROCESS HISTORY
    # =================================================================
    if calibration_steps:
        # Join steps with proper separator for astronomy software compatibility
        steps_str = ' -> '.join(calibration_steps)
        header['CALSTEPS'] = (steps_str, 'Calibration steps applied in order')
        header['NSTEPS'] = (len(calibration_steps), 'Number of calibration steps')
        
        # Add individual step details for machine readability
        for i, step in enumerate(calibration_steps, 1):
            if i <= 9:  # FITS keyword limit
                header[f'STEP{i:01d}'] = (step, f'Calibration step {i}')
    
    # =================================================================
    # MASTER FRAME REFERENCES AND METADATA
    # =================================================================
    master_count = 0
    
    if bias_master and os.path.exists(bias_master):
        master_count += 1
        header['BIASMAST'] = (os.path.basename(bias_master), 'Master bias frame filename')
        header['BIASREF'] = (bias_master, 'Full path to master bias frame')
        
        # Add master frame checksum for verification
        try:
            with open(bias_master, 'rb') as f:
                bias_hash = hashlib.md5(f.read()).hexdigest()[:16]  # Truncate for FITS
            header['BIASMD5'] = (bias_hash, 'MD5 checksum of bias master (truncated)')
        except:
            pass
            
        # Try to get master frame creation info
        try:
            from astropy.io import fits
            with fits.open(bias_master) as hdul:
                bias_header = hdul[0].header
                if 'CREATED' in bias_header:
                    header['BIASMADE'] = (bias_header['CREATED'], 'Bias master creation date')
                if 'NFRAMES' in bias_header:
                    header['BIASN'] = (bias_header['NFRAMES'], 'Number of frames in bias master')
        except:
            pass
    
    if dark_master and os.path.exists(dark_master):
        master_count += 1
        header['DARKMAST'] = (os.path.basename(dark_master), 'Master dark frame filename')
        header['DARKREF'] = (dark_master, 'Full path to master dark frame')
        
        try:
            with open(dark_master, 'rb') as f:
                dark_hash = hashlib.md5(f.read()).hexdigest()[:16]
            header['DARKMD5'] = (dark_hash, 'MD5 checksum of dark master (truncated)')
        except:
            pass
            
        try:
            from astropy.io import fits
            with fits.open(dark_master) as hdul:
                dark_header = hdul[0].header
                if 'CREATED' in dark_header:
                    header['DARKMADE'] = (dark_header['CREATED'], 'Dark master creation date')
                if 'NFRAMES' in dark_header:
                    header['DARKN'] = (dark_header['NFRAMES'], 'Number of frames in dark master')
                if 'EXPTIME' in dark_header:
                    header['DARKEXP'] = (dark_header['EXPTIME'], 'Dark master exposure time')
        except:
            pass
    
    if flat_master and os.path.exists(flat_master):
        master_count += 1
        header['FLATMAST'] = (os.path.basename(flat_master), 'Master flat frame filename')
        header['FLATREF'] = (flat_master, 'Full path to master flat frame')
        
        try:
            with open(flat_master, 'rb') as f:
                flat_hash = hashlib.md5(f.read()).hexdigest()[:16]
            header['FLATMD5'] = (flat_hash, 'MD5 checksum of flat master (truncated)')
        except:
            pass
            
        try:
            from astropy.io import fits
            with fits.open(flat_master) as hdul:
                flat_header = hdul[0].header
                if 'CREATED' in flat_header:
                    header['FLATMADE'] = (flat_header['CREATED'], 'Flat master creation date')
                if 'NFRAMES' in flat_header:
                    header['FLATN'] = (flat_header['NFRAMES'], 'Number of frames in flat master')
                if 'FILTER' in flat_header:
                    header['FLATFILT'] = (flat_header['FILTER'], 'Flat master filter')
        except:
            pass
    
    header['NMASTERS'] = (master_count, 'Number of master frames applied')
    
    # =================================================================
    # CALIBRATION QUALITY METRICS
    # =================================================================
    # Basic image statistics
    header['CALMEAN'] = (float(np.mean(calibrated_data)), 'Mean pixel value after calibration')
    header['CALMED'] = (float(np.median(calibrated_data)), 'Median pixel value after calibration')
    header['CALSTD'] = (float(np.std(calibrated_data)), 'Standard deviation after calibration')
    header['CALNOISE'] = (float(np.std(calibrated_data)), 'Noise level (RMS) after calibration')
    
    # Dynamic range and signal metrics
    min_val = float(np.min(calibrated_data))
    max_val = float(np.max(calibrated_data))
    header['CALMIN'] = (min_val, 'Minimum pixel value after calibration')
    header['CALMAX'] = (max_val, 'Maximum pixel value after calibration')
    header['CALRANGE'] = (max_val - min_val, 'Dynamic range after calibration')
    
    # Signal-to-noise estimation
    if np.std(calibrated_data) > 0:
        snr_estimate = np.mean(calibrated_data) / np.std(calibrated_data)
        header['CALSNR'] = (float(snr_estimate), 'Estimated signal-to-noise ratio')
    
    # Hot/dead pixel detection
    data_flat = calibrated_data.flatten()
    sorted_data = np.sort(data_flat)
    p99_9 = sorted_data[int(len(sorted_data) * 0.999)]
    p0_1 = sorted_data[int(len(sorted_data) * 0.001)]
    
    hot_pixels = np.sum(calibrated_data > p99_9)
    dead_pixels = np.sum(calibrated_data < p0_1)
    
    header['CALHOT'] = (int(hot_pixels), 'Number of potential hot pixels')
    header['CALDEAD'] = (int(dead_pixels), 'Number of potential dead pixels')
    
    # =================================================================
    # PROCESSING ENVIRONMENT AND PARAMETERS
    # =================================================================
    header['CALHOST'] = (os.environ.get('COMPUTERNAME', 'Unknown'), 'Computer used for calibration')
    header['CALOS'] = (f"{os.name}", 'Operating system used for calibration')
    
    # Processing parameters
    header['CALNEG'] = (bool(np.any(calibrated_data < 0)), 'Negative values present after calibration')
    header['CALCLIP'] = ('Applied' if np.any(calibrated_data < 0) else 'None', 'Negative value clipping applied')
    
    # =================================================================
    # ASTRONOMY SOFTWARE COMPATIBILITY HEADERS
    # =================================================================
    # Common headers expected by popular astronomy software
    header['PROCESSED'] = (True, 'Frame has been processed/calibrated')
    header['REDUCER'] = ('AstroFiler', 'Software used for calibration')
    header['REDDATE'] = (datetime.now().strftime('%Y-%m-%d'), 'Date of calibration processing')
    
    # Compatibility with common pipeline formats
    header['HISTORY'] = f"CALIBRATED by AstroFiler v1.2.0 on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    if calibration_steps:
        header['HISTORY'] = f"Applied: {' -> '.join(calibration_steps)}"
    
    # =================================================================
    # DATA INTEGRITY AND VERIFICATION
    # =================================================================
    header['CALSUM'] = (float(np.sum(calibrated_data)), 'Checksum: sum of all pixel values')
    header['CALN'] = (int(calibrated_data.size), 'Total number of pixels')
    
    # Version tracking for future compatibility
    header['CALVER'] = ('1.0', 'Calibration header format version')


In [68]:
def calibrate_light_frame(light_path: str, dark_master: Optional[str] = None, 
                         flat_master: Optional[str] = None, bias_master: Optional[str] = None, 
                         output_path: Optional[str] = None, progress_callback: Optional[Callable] = None) -> Dict:
    try:
        if progress_callback:
            progress_callback(f"Starting calibration of {os.path.basename(light_path)}")
            
        # Validate input file
        if not os.path.exists(light_path):
            return {"error": f"Light frame file not found: {light_path}"}
        
        if progress_callback:
            progress_callback("Loading light frame...")
            
        with fits.open(light_path) as hdul:
            original_data = None
            light_header = None
            for hdu in hdul:
                if hdu.data is not None and hdu.data.ndim >= 2:
                    original_data = hdu.data
                    light_header = hdu.header.copy()
                    break
            if light_header is None:
                light_header = hdul[0].header.copy()
            if original_data is None:
                return {"error": "No image data found in light frame"}
            light_data = original_data.astype(np.float64)
        
        if light_data.size == 0:
            return {"error": "No image data found in light frame"}
        
        calibrated_data = light_data.copy()
        calibration_steps = []
        
        # =================================================================
        # STEP 1: DARK SUBTRACTION
        # Formula: calibrated = Light - (Dark - Bias)
        # =================================================================
      
        if dark_master and os.path.exists(dark_master):
            if progress_callback:
                progress_callback("Applying dark correction...")
            try:
                with fits.open(dark_master) as hdul:
                    _dark_raw, dark_header = _fits_image_data(hdul)
                    if _dark_raw is None:
                        raise ValueError("No image data found in dark master")
                    dark_data = _dark_raw.astype(np.float64)

                    # Load bias and subtract from dark
                    bias_data = None
                    if bias_master and os.path.exists(bias_master):
                        try:
                            with fits.open(bias_master) as bias_hdul:
                                _bias_raw, _ = _fits_image_data(bias_hdul)
                                if _bias_raw is None:
                                    raise ValueError("No image data found in bias master")
                                bias_data = _bias_raw.astype(np.float64)
                                print(f"Bias loaded: shape={bias_data.shape}, mean={np.mean(bias_data):.2f}")
                        except Exception as e:
                            print(f"WARNING: Failed to load bias: {e} — using uncorrected dark")

                    if bias_data is not None and bias_data.shape == dark_data.shape:
                        dark_corrected = dark_data - bias_data
                        calibration_steps.append(f"DARK-BIAS: {os.path.basename(dark_master)}")
                        print(f"Dark - Bias applied: dark_mean={np.mean(dark_data):.2f}, bias_mean={np.mean(bias_data):.2f}, corrected_mean={np.mean(dark_corrected):.2f}")
                    else:
                        dark_corrected = dark_data
                        calibration_steps.append(f"DARK (no bias): {os.path.basename(dark_master)}")
                        print("WARNING: No bias correction applied to dark")

                    before_mean = np.mean(calibrated_data)
                    calibrated_data = calibrated_data - dark_corrected
                    after_mean = np.mean(calibrated_data)
                    print(f"Dark subtracted: before_mean={before_mean:.2f}, after_mean={after_mean:.2f}, delta={before_mean - after_mean:.2f}")
                    
            except Exception as e:
                print(f"ERROR: Failed to apply dark correction: {e}")
                import traceback; traceback.print_exc()

        # =================================================================
        # GENERATE OUTPUT PATH
        # =================================================================
        
        if not output_path:
            base_dir = os.path.dirname(light_path)
            base_name = os.path.basename(light_path)
            output_path = os.path.join(base_dir, f"cal_{base_name}")
        
        # =================================================================
        # SAVE CALIBRATED FRAME
        # =================================================================
        
        if progress_callback:
            progress_callback("Saving calibrated frame...")
            
        data_min = float(np.min(calibrated_data))
        data_max = float(np.max(calibrated_data))
        data_mean = float(np.mean(calibrated_data))
        data_std = float(np.std(calibrated_data))
        
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        hdu = fits.PrimaryHDU(data=calibrated_data.astype(np.float32), header=light_header)
        hdu.writeto(output_path, overwrite=True)
        
        if progress_callback:
            progress_callback(f"Calibration complete: {os.path.basename(output_path)}")
            
        return {
            "success": True,
            "output_path": output_path,
            "input_path": light_path,
            "calibration_steps": calibration_steps,
            "method": "Numpy",
            "noise_level": data_std,
            "mean_level": data_mean,
            "dynamic_range": data_max - data_min,
            "data_min": data_min,
            "data_max": data_max
        }
        
    except Exception as e:
        import traceback; traceback.print_exc()
        return {"error": f"Failed to calibrate light frame: {str(e)}"}


In [69]:
def main():
    print("Running light frame calibration test...")
    _folder="C:/REPOSITORY.test/Light/M_101/Celestron_C8_2032@F_10.0/ZWO_CCD_ASI183MM_Pro/20240218"
    _file = "M_101-Celestron_C8_2032@F_10.0-ZWO_CCD_ASI183MM_Pro-Ha-20240218085110-60.0s-3x3-t-25.0.fits"
    _master_folder = "C:/REPOSITORY.test/Masters"
    _master_dark = f"{_master_folder}/Master-Dark-Celestron_C8_2032_F_10.0-ZWO_CCD_ASI183MM_Pro-20240116-60.0s-3x3-t-25.0.fits"
    _master_flat = f"{_master_folder}/Master-Flat-Celestron_C8_2032_F_10.0-ZWO_CCD_ASI183MM_Pro-Ha-20240116-18.07675s-3x3-t-25.0.fits"
    _master_bias = f"{_master_folder}/Master-Bias-Celestron_C8_2032_F_10.0-ZWO_CCD_ASI183MM_Pro-20240116-3.2e-05s-3x3-t-25.0.fits"
    
    result = calibrate_light_frame(
        light_path=f"{_folder}/{_file}",
        dark_master=f"{_master_folder}/{_master_dark}",
        flat_master=f"{_master_folder}/{_master_flat}",
        bias_master=f"{_master_folder}/{_master_bias}",
    )
    if result.get('success'):
        print(f"Calibrated frame saved: {result['output_path']}")
    else:
        print(f"Calibration failed: {result.get('error')}")

In [70]:
main()

Running light frame calibration test...
Calibrated frame saved: C:/REPOSITORY.test/Light/M_101/Celestron_C8_2032@F_10.0/ZWO_CCD_ASI183MM_Pro/20240218\cal_M_101-Celestron_C8_2032@F_10.0-ZWO_CCD_ASI183MM_Pro-Ha-20240218085110-60.0s-3x3-t-25.0.fits
